In [1]:
import h5py
import numpy as np

In [2]:
def print_structure(f, indent=0):
    for key in f.keys():
        item = f[key]
        print(' ' * indent + key, '—', type(item).__name__,
              getattr(item, 'shape', ''))
        if hasattr(item, 'keys'):
            print_structure(item, indent + 4)

with h5py.File('/Users/suthardr/Desktop/friends_rachel_day2_06162026.mesc', 'r') as f:
    print_structure(f)


MSession_0 — Group 
    MUnit_0 — Group 
        Channel_0 — Dataset (1, 384, 516)
    MUnit_4 — Group 
        Channel_0 — Dataset (1, 656, 656)
        Channel_1 — Dataset (1, 656, 656)
    MUnit_5 — Group 
        Channel_0 — Dataset (1, 656, 656)
        Channel_1 — Dataset (1, 656, 656)
    MUnit_6 — Group 
        Channel_0 — Dataset (21630, 656, 656)
        Channel_1 — Dataset (21630, 656, 656)
        Curve_0 — Group 
            CurveDataYIdxNextSample — Dataset (1,)
            CurveDataYRawData — Dataset (1,)
        Curve_1 — Group 
            CurveDataYIdxNextSample — Dataset (21631,)
            CurveDataYRawData — Dataset (21631,)
        Curve_2 — Group 
            CurveDataYIdxNextSample — Dataset (2,)
            CurveDataYRawData — Dataset (2,)
        Curve_3 — Group 
            CurveDataYIdxNextSample — Dataset (108143,)
            CurveDataYRawData — Dataset (108143,)
        Curve_4 — Group 
            CurveDataYIdxNextSample — Dataset (108140,)
           

In [ ]:
import h5py
import numpy as np
import tifffile
import json
import re
from pathlib import Path

MESC_FOLDER = Path('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day4-RecallToneB')
OUT_ROOT    = Path('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day4-RecallToneB')

# ── sampling constants (fixed for this cohort) ───────────────────────────────
CURVE_SAMPLE_DT_MS = 0.06484
FRAME_DT_MS        = 42.535039999999995
TRUE_FRAME_DT_MS   = FRAME_DT_MS * 2
TRUE_FRAME_RATE    = 1000.0 / TRUE_FRAME_DT_MS
CH0_T0_MS          = 0.0
CH1_T0_MS          = FRAME_DT_MS

# ── helpers ───────────────────────────────────────────────────────────────────
def find_imaging_unit(f):
    best_path, best_frames, best_shape = None, 0, None
    def visit(name, obj):
        nonlocal best_path, best_frames, best_shape
        if isinstance(obj, h5py.Dataset) and 'Channel_0' in name:
            shape = obj.shape
            if len(shape) == 3 and shape[0] > best_frames:
                best_frames = shape[0]
                best_path   = name.replace('/Channel_0', '')
                best_shape  = shape
    f.visititems(visit)
    return best_path, best_frames, best_shape


def parse_filename(stem):
    match = re.match(r'friends_(\w+)_(day\d+)_(\d{8})', stem)
    if match:
        subject  = match.group(1)
        session  = match.group(2)
        date_raw = match.group(3)
        date     = f'20{date_raw[4:6]}-{date_raw[0:2]}-{date_raw[2:4]}'
    else:
        subject, session, date = stem, 'unknown', 'unknown'
    return subject, session, date


def process_file(mesc_path):
    stem                     = mesc_path.stem
    subject, session, date   = parse_filename(stem)
    OUT_DIR                  = OUT_ROOT / f'{stem}_extracted'
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f'\n{"═"*70}')
    print(f'  {subject.upper()}  |  {session}  |  {date}')
    print(f'  {mesc_path.name}')
    print(f'  → {OUT_DIR}')
    print(f'{"═"*70}')

    with h5py.File(mesc_path, 'r') as f:
        UNIT_PATH, N_FRAMES, frame_shape = find_imaging_unit(f)

        if UNIT_PATH is None:
            print('  ⚠ no imaging MUnit found — skipping')
            return

        TRUE_N_FRAMES = N_FRAMES // 2
        H, W          = frame_shape[1], frame_shape[2]

        print(f'  Imaging unit:  {UNIT_PATH}')
        print(f'  Stored frames: {N_FRAMES}  →  {TRUE_N_FRAMES} true frames')
        print(f'  Frame size:    {H} × {W} px  |  {TRUE_FRAME_RATE:.2f} Hz\n')

        unit = f[UNIT_PATH]

        # ── 1. metadata ──────────────────────────────────────────────────────
        meta = {
            'file':    str(mesc_path),
            'subject': subject,
            'session': session,
            'date':    date,
            'munit': {
                'path':                       UNIT_PATH,
                'section_type':               str(unit.attrs.get('SectionTypeDebugString', 'unknown')),
                'technology':                 str(unit.attrs.get('TechnologyTypeDebugString', 'unknown')),
                'scanner':                    str(unit.attrs.get('Scanner', 'unknown')),
                'n_channels':                 int(unit.attrs.get('VecChannelsSize', 2)),
                'stored_frame_period_ms':     FRAME_DT_MS,
                'stored_frame_rate_hz':       1000.0 / FRAME_DT_MS,
                'stored_n_frames':            N_FRAMES,
                'true_frame_period_ms':       TRUE_FRAME_DT_MS,
                'true_frame_rate_hz':         TRUE_FRAME_RATE,
                'true_n_frames':              TRUE_N_FRAMES,
                'frame_size_px':              [H, W],
                'pmt_gating':                 'interleaved_channels',
                'channel_0_active_parity':    'even (0-indexed)',
                'channel_1_active_parity':    'odd (0-indexed)',
                'channel_0_t0_ms':            CH0_T0_MS,
                'channel_1_t0_ms':            CH1_T0_MS,
                'channel_temporal_offset_ms': FRAME_DT_MS,
                'T0_ms':                      float(unit.attrs.get('T0InMs', 0.0)),
                'precalc_time_ms':            float(unit.attrs.get('PrecalculationTimeInMs', 0.0)),
                'mes_version':                json.loads(unit.attrs['VersionInfoJSON'])['mesVersion']
                                              if 'VersionInfoJSON' in unit.attrs else 'unknown',
                'uuid':                       unit.attrs['Uuid'].tolist()
                                              if 'Uuid' in unit.attrs else None,
            },
            'channels': {},
        }

        for ch_name in ['Channel_0', 'Channel_1']:
            if ch_name in unit:
                meta['channels'][ch_name] = {
                    k: (v.tolist() if hasattr(v, 'tolist') else str(v))
                    for k, v in dict(unit[ch_name].attrs).items()
                }

        with open(OUT_DIR / 'metadata.json', 'w') as fj:
            json.dump(meta, fj, indent=2, default=str)
        print(f'  ✓ metadata.json')

        # ── 2. frame timestamps ──────────────────────────────────────────────
        t0                 = float(unit.attrs.get('T0InMs', 0.0))
        ch0_frame_times_ms = t0 + CH0_T0_MS + np.arange(TRUE_N_FRAMES) * TRUE_FRAME_DT_MS
        ch1_frame_times_ms = t0 + CH1_T0_MS + np.arange(TRUE_N_FRAMES) * TRUE_FRAME_DT_MS
        np.save(OUT_DIR / 'frame_times_ms_green.npy', ch0_frame_times_ms)
        np.save(OUT_DIR / 'frame_times_ms_red.npy',   ch1_frame_times_ms)
        print(f'  ✓ frame_times_ms_green.npy  '
              f'({TRUE_N_FRAMES} frames, {ch0_frame_times_ms[-1]/1000:.1f} s)')
        print(f'  ✓ frame_times_ms_red.npy    '
              f'(offset +{FRAME_DT_MS:.3f} ms from green)')

        # ── 3. imaging data ──────────────────────────────────────────────────
        channel_parity = {
            'Channel_0': (0, 'even', 'green'),
            'Channel_1': (1, 'odd',  'red'),
        }

        for ch_name, (parity, parity_label, color) in channel_parity.items():
            if ch_name not in unit:
                print(f'  skipping {ch_name} — not found')
                continue

            print(f'\n  Loading {ch_name} ({color})...')
            raw  = unit[ch_name][:]
            data = raw[parity::2]

            active_mean  = raw[parity::2].mean()
            blanked_mean = raw[1 - parity::2].mean()
            ratio        = active_mean / blanked_mean if blanked_mean > 0 else float('inf')

            if ratio > 1.05:
                print(f'  ✓ parity check passed  '
                      f'(active {active_mean:.1f}  blanked {blanked_mean:.1f}  '
                      f'ratio {ratio:.2f}x)')
            else:
                print(f'  ⚠ parity check FAILED  '
                      f'(active {active_mean:.1f}  blanked {blanked_mean:.1f}  '
                      f'ratio {ratio:.2f}x) — verify visually')

            t0_ms    = CH0_T0_MS if parity == 0 else CH1_T0_MS
            out_path = OUT_DIR / f'channel_{color}.tif'
            tifffile.imwrite(
                out_path,
                data,
                bigtiff=True,
                metadata={
                    'axes': 'TYX',
                    'fps':  TRUE_FRAME_RATE,
                    'unit': 'um',
                    'Info': json.dumps({
                        'subject':               subject,
                        'session':               session,
                        'date':                  date,
                        'channel':               ch_name,
                        'color':                 color,
                        'frame_rate_hz':         TRUE_FRAME_RATE,
                        'frame_period_ms':       TRUE_FRAME_DT_MS,
                        'n_frames':              TRUE_N_FRAMES,
                        'n_frames_raw':          N_FRAMES,
                        'pmt_gating':            'interleaved_channels',
                        'active_frame_parity':   parity_label + ' (0-indexed)',
                        't0_ms':                 t0_ms,
                        'source_file':           str(mesc_path),
                        'source_path':           UNIT_PATH + '/' + ch_name,
                    })
                }
            )
            print(f'  ✓ channel_{color}.tif  '
                  f'shape={data.shape}  dtype={data.dtype}  '
                  f'{out_path.stat().st_size/1e9:.2f} GB')

    print(f'\n  Output files:')
    for p in sorted(OUT_DIR.iterdir()):
        print(f'    {p.name:<45}  {p.stat().st_size/1e6:8.1f} MB')


# ── main loop ─────────────────────────────────────────────────────────────────
mesc_files = sorted(MESC_FOLDER.glob('*.mesc'))

if not mesc_files:
    print(f'No .mesc files found in {MESC_FOLDER}')
else:
    print(f'Found {len(mesc_files)} .mesc files:')
    for f in mesc_files:
        print(f'  {f.name}')

    failed = []
    for i, mesc_path in enumerate(mesc_files):
        print(f'\n[{i+1}/{len(mesc_files)}] {mesc_path.name}')
        try:
            process_file(mesc_path)
        except Exception as e:
            print(f'  ✗ FAILED: {e}')
            failed.append((mesc_path.name, str(e)))

    print(f'\n{"═"*70}')
    print(f'Done.  {len(mesc_files) - len(failed)}/{len(mesc_files)} files processed successfully.')
    if failed:
        print('Failed:')
        for name, err in failed:
            print(f'  {name}: {err}')

Found 5 .mesc files:
  friends_chandler_day4_06182026.mesc
  friends_monica_day4_06182026.mesc
  friends_phoebe_day4_06182026.mesc
  friends_phoebe_day4_TESTONSCANSPEED4.mesc
  friends_rachel_day4_06182026.mesc

[1/5] friends_chandler_day4_06182026.mesc

══════════════════════════════════════════════════════════════════════
  CHANDLER  |  day4  |  2020-06-18
  friends_chandler_day4_06182026.mesc
  → /Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day4-RecallToneB/friends_chandler_day4_06182026_extracted
══════════════════════════════════════════════════════════════════════
  Imaging unit:  MSession_0/MUnit_8
  Stored frames: 21630  →  10815 true frames
  Frame size:    656 × 656 px  |  11.76 Hz

  ✓ metadata.json
  ✓ frame_times_ms_green.npy  (10815 frames, 919.9 s)
  ✓ frame_times_ms_red.npy    (offset +42.535 ms from green)

  Loading Channel_0 (green)...
